In [1]:
# Duplicates clearing
train_df = train_df.drop_duplicates(subset=feature_columns).reset_index(drop=True)

# Constant columns and the identifier are not used in training data
drop_cols = ["ID", "Z_CostContact", "Z_Revenue"]
train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)

# Clear data errors found in Week 2 (training set only/ Income n/a values)
train_df = train_df[(train_df["Year_Birth"] >= 1940) & (train_df["Income"].fillna(0) < 600000)]

NameError: name 'train_df' is not defined

In [2]:
# Merging odd labels
for d in (train_df, test_df):
    for col in ["Education", "Marital_Status"]:
        d[col] = d[col].str.strip().str.lower()
    d["Marital_Status"] = d["Marital_Status"].replace(
        {"alone": "single", "absurd": "single", "yolo": "single",
         "together": "married"})   
    d["Education"] = d["Education"].replace({"2n cycle": "master"})

NameError: name 'train_df' is not defined

In [ ]:
# Categorizing / bucketizing
ref_year = 2014   # the data ends in mid-2014
ref_date = pd.to_datetime(train_df["Dt_Customer"], format="%d-%m-%Y").max()

for d in (train_df, test_df):
    d["Age"] = ref_year - d["Year_Birth"]
    d["Tenure_Days"] = (ref_date - pd.to_datetime(d["Dt_Customer"], format="%d-%m-%Y")).dt.days
    d["Age_Group"] = pd.cut(d["Age"], bins=[0, 35, 50, 65, 120],
                            labels=["<35", "35-49", "50-64", "65+"])
    d["Total_Spend"] = d[spend_columns].sum(axis=1)

# Quartile cut points learned from train, reused on test
income_bins = train_df["Income"].quantile([0, .25, .5, .75, 1]).values
income_bins[0], income_bins[-1] = -np.inf, np.inf
for d in (train_df, test_df):
    d["Income_Band"] = pd.cut(d["Income"], bins=income_bins, labels=["Q1", "Q2", "Q3", "Q4"])

train_df = train_df.drop(columns=["Dt_Customer"])
test_df = test_df.drop(columns=["Dt_Customer"])

In [ ]:
# Scaling / one-
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

skewed = spend_columns + ["Income", "Total_Spend"]   # heavy right skew from EDA
other_num = ["Age", "Tenure_Days", "Kidhome", "Teenhome", "Recency",
             "NumDealsPurchases", "NumWebPurchases", "NumCatalogPurchases",
             "NumStorePurchases", "NumWebVisitsMonth"]
categorical = ["Education", "Marital_Status", "Age_Group", "Income_Band"]

preprocess = ColumnTransformer([
    ("skewed", Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("log", FunctionTransformer(np.log1p)),
                         ("scale", StandardScaler())]), skewed),
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), other_num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical),
], remainder="passthrough")   # keeps binary columns like Complain and AcceptedCmp1-5

X_train = train_df.drop(columns="Response")
X_test = test_df.drop(columns="Response")

X_train_p = preprocess.fit_transform(X_train)
X_test_p = preprocess.transform(X_test)